# Skin-Lesion Classification on HAM10000: CNN Baseline, Transfer Learning and a Critical Comparison with DiffMIC

**CM4126 Computer Vision, Coursework 2026/27** | Dataset: [HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) | Reviewed paper: *[DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification](https://link.springer.com/chapter/10.1007/978-3-031-43987-2_10)* (MICCAI 2023)

## Contents

1. [Introduction](#1-introduction)
2. [Research and Data Exploration](#2-research-and-data-exploration): paper review, dataset choice, exploration, splits, image quality, enhancement
3. [Baseline Model](#3-baseline-model): training framework and a CNN trained from scratch
4. [Solution Improvement](#4-solution-improvement): transfer learning (3 pretrained models), class imbalance, image enhancement, GAN augmentation, plus a hyper-parameter search and a seed study
5. [Evaluation, Visualisation and Comparison](#5-evaluation-visualisation-and-comparison): test-set results, Grad-CAM, comparison with the baseline and with the paper
6. [Conclusion and Future Directions](#6-conclusion-and-future-directions)
7. [References](#references) and [Appendix](#appendix)

## 1. Introduction

**Problem.** Skin cancer is one of the most common cancers, and melanoma is its most dangerous form: caught early it is usually curable, caught late it is often fatal. A classifier for dermatoscopic images could act as a second reader and help prioritise patients.

**Task.** Classify a dermascopic image into one of 7 diagnostic categories (for example naevus, melanoma or basal cell carcinoma) on a highly imbalanced dataset, HAM10000.

**Objectives:**

| # | Objective | Where |
|---|---|---|
| 1 | Select and critically review a high-quality paper | §2.1 |
| 2 | Justify and describe the dataset (size, splits, classes, image quality, standardisation, enhancement) | §2.2–§2.10 |
| 3 | Build a CNN **baseline** that follows good practice, with every setting justified | §3 |
| 4 | **Improve** it: transfer learning (three pretrained models), imbalance handling, image enhancement, GAN images, a hyper-parameter search and a seed study | §4 |
| 5 | Evaluate with suitable metrics, **visualise** the results (learning curves, confusion matrices, Grad-CAM) and **compare** them with the baseline and the paper | §5 |
| 6 | Keep everything **reproducible** | whole notebook |

**Principles I followed throughout:**

1. **No data leakage.** HAM10000 often has several photos of one lesion, so I split the data by lesion, not by image. The paper's random image split is reproduced once to measure what it changes (§4.8, §5.6).
2. **Decisions use the validation set only.** The imbalance strategy, enhancement, hyper-parameters, early stopping and checkpoints never look at the test set, which is used only in §3.3 and §5.
3. **Accuracy is not enough.** 67 % of the images are one class, so always answering "naevus" is already 67 % accurate. The main metrics are **macro-F1** (the same family as the paper's F1), **balanced accuracy**, **melanoma recall** and one-vs-rest **AUC**. Per class I report recall, precision and F1.

### How to run this notebook

Open the notebook in VS Code, follow the set-up steps in the first code cell (§1.1) and press **Run All**; there is a single configuration.

* **Data:** if HAM10000 is not in `data/` yet, the first cells download it from Kaggle.
* **Saved results:** resized images and trained models are kept in `outputs/` and reloaded only if their settings and data are unchanged. Deleting `outputs/` forces a full re-run, which takes about an hour on the reference machine.
* **Reference machine:** AMD Ryzen 7 7800X3D, 64 GB RAM, AMD Radeon RX 7900 XT (20 GB), Windows 11, Python 3.12.10, PyTorch 2.9.1 (ROCm 7.2.1); speed benchmark in Appendix B.
* **Reproducibility:** every random generator uses seed 42 and the GPU kernels are deterministic. Another machine or driver can still give slightly different numbers, within the run-to-run noise measured in §4.6.

### 1.1 Set-up and configuration

All settings live in one cell, and each value is explained where it is used.

A few engineering choices make the notebook faster without changing what the models learn: a workaround for a BatchNorm bug in the Windows ROCm build of PyTorch (Appendix A); decoding the resized images once, caching them on disk and keeping them on the GPU, with augmentation on the GPU as well, because the CPU data loader was the bottleneck (Appendix B); 16-bit mixed precision, which was roughly two to five times faster than 32-bit in the benchmark; and saving trained models so that they are reused when their settings have not changed.

In [ ]:
#Set-up: create a virtual environment with Python 3.12 (e.g. `python -m venv venv`) and select it as the notebook kernel.
#Then, BEFORE running this cell, install PyTorch and torchvision built for your hardware:
#  NVIDIA, Linux:   `pip install torch torchvision` (the default build already supports CUDA).
#  NVIDIA, Windows: the default build is CPU-only; use the CUDA install command from https://pytorch.org/get-started/locally/
#  AMD, Linux:      use the ROCm install command from https://pytorch.org/get-started/locally/
#  AMD, Windows:    use AMD's ROCm build of PyTorch (see AMD's ROCm documentation); this is the set-up used here (Appendix A).
#  macOS:           `pip install torch torchvision`; current releases use the GPU only on Apple-silicon Macs (MPS), not NVIDIA or AMD.
#ROCm builds expose the GPU through torch.cuda, so NVIDIA and AMD GPUs are found the same way. Without a GPU the notebook still runs, but training takes far longer.
%pip install -q numpy pandas matplotlib opencv-python pillow scikit-learn tqdm kagglehub

In [ ]:
import os, platform, copy, hashlib, json, math, random, sys, time, warnings
from functools import partial
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Windows + AMD only: skip a GPU-detection probe that fails when the folder path contains spaces.
# Has to happen before torch is imported (Appendix A).
if os.name == "nt" and "ROCM_SDK_TARGET_FAMILY" not in os.environ:
    try:
        from rocm_sdk import _dist_info
        if "custom" in _dist_info.AVAILABLE_TARGET_FAMILIES:
            os.environ["ROCM_SDK_TARGET_FAMILY"] = "custom"
    except Exception:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models as tvm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from tqdm import tqdm as _tqdm
from IPython.display import display

tqdm = partial(_tqdm, file=sys.stdout, ncols=100, mininterval=1)      # plain-text progress bars, saved with the notebook

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30, "display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})
print(f"Python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"numpy {np.__version__} | pandas {pd.__version__} | opencv {cv2.__version__}")

In [ ]:
# Settings shared by all experiments
SEED = 42

# images
IMG_SIZE = 224                           # input size of the ImageNet models
IMAGENET_MEAN = (0.485, 0.456, 0.406)    # ImageNet statistics, expected by the pretrained weights
IMAGENET_STD = (0.229, 0.224, 0.225)
N_CLASSES = 7

# training
BATCH_SIZE = 32                          # throughput stops rising here on this GPU (Appendix B)
EPOCHS_CNN = 20                          # baseline; early stopping usually ends it sooner
EPOCHS_HEAD = 3                          # transfer learning stage 1: new output layer only
EPOCHS_FINETUNE = 25                     # transfer learning stage 2: whole network
PATIENCE = 5                             # early stopping, pretrained models
PATIENCE_CNN = 5                         # the baseline learns more slowly
N_SEARCH_TRIALS = 3
SEARCH_FT_EPOCHS = 3                     # kept shot on purpose
GAN_STEPS = 5000                        # about 134 passes over the training images at batch 64; not tuned
N_BOOTSTRAP = 200                       # resamples for the 95 % confidence intervals

# reported by the paper on HAM10000 (Table 1b)
PAPER = {"name": "DiffMIC (Yang et al., MICCAI 2023)", "accuracy": 0.906, "f1": 0.816}

In [ ]:
# Download HAM10000 from Kaggle into data/ unless it is already there
import shutil, kagglehub
DATA_DIR = Path("data")
if not (DATA_DIR / "HAM10000_metadata.csv").exists():
    kaggle_dir = Path(kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000"))
    DATA_DIR.mkdir(exist_ok=True)
    for name in ["HAM10000_metadata.csv", "hmnist_8_8_L.csv", "hmnist_8_8_RGB.csv", "hmnist_28_28_L.csv", "hmnist_28_28_RGB.csv"]:
        shutil.copy2(kaggle_dir / name, DATA_DIR / name)
    for name in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        shutil.copytree(kaggle_dir / name, DATA_DIR / name, dirs_exist_ok=True)
print("data jpg files:", len(list(DATA_DIR.rglob("*.jpg"))))

In [ ]:
DATA_DIR = Path("data")
OUT_DIR = Path("outputs/checkpoints")    # trained weights and training histories
CACHE_DIR = Path("outputs/cache")        # resized images (.npy)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

META_CSV = DATA_DIR / "HAM10000_metadata.csv"
IMAGE_PATHS = {p.stem: p for p in DATA_DIR.rglob("*.jpg")}      # image_id -> file
print("metadata  :", META_CSV.resolve())
print("jpg files :", len(IMAGE_PATHS))

In [ ]:
def set_seed(seed):
    """Seed every random generator and make the GPU kernels deterministic."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


if torch.cuda.is_available():                      # AMD ROCm also shows up as "cuda"
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Mixed precision (Appendix B). float16 needs a GradScaler; bfloat16 does not and is safer for transformers, so ViT uses it.
AMP_FP16 = torch.float16 if device.type == "cuda" else None
AMP_BF16 = torch.bfloat16 if device.type == "cuda" else None

set_seed(SEED)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
# Windows + AMD only (Appendix A): the ROCm build sends BatchNorm to MIOpen, whose run-time kernel compile fails with
# "'type_traits' file not found" (https://github.com/ROCm/TheRock/issues/9181). PyTorch's own GPU kernel does the same maths.
def apply_rocm_windows_batchnorm_fix():
    if os.name != "nt" or getattr(torch.version, "hip", None) is None:
        return False
    original = F.batch_norm

    def native_batch_norm(input, running_mean, running_var, weight=None, bias=None, training=False, momentum=0.1, eps=1e-5):
        if not input.is_cuda:
            return original(input, running_mean, running_var, weight, bias, training, momentum, eps)
        if training and input.numel() // max(input.shape[1], 1) <= 1:
            raise ValueError(f"Expected more than 1 value per channel when training, got {tuple(input.shape)}")
        previous = torch.backends.cudnn.enabled         # PyTorch 2.9 chooses the backend from this flag
        torch.backends.cudnn.enabled = False
        try:
            return torch.batch_norm(input, weight, bias, running_mean, running_var, training, momentum, eps, False)
        finally:
            torch.backends.cudnn.enabled = previous

    F.batch_norm = native_batch_norm
    return True


print("BatchNorm workaround for Windows + ROCm active:", apply_rocm_windows_batchnorm_fix())

In [ ]:
# Numbered captions: table captions go above the table, figure captions below the figure.
from IPython.display import Markdown

_CAPTION_COUNTER = {"Figure": 0, "Table": 0}


def caption(kind, text):
    _CAPTION_COUNTER[kind] += 1
    display(Markdown(f"**{kind} {_CAPTION_COUNTER[kind]}.** {text}"))


def show_fig(text):
    plt.show()
    caption("Figure", text)


def show_table(df, text):
    caption("Table", text)
    display(df)